## **3. Importación de librerías**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

## **4. Configuración de Pandas**

In [2]:
# Mostrar todas las columnas
pd.set_option('display.max_columns', None)

# Mostrar todo el texto en las celdas sin truncar
pd.set_option('display.max_colwidth', None)

## **5. Carga de datos**

In [3]:
# Cargamos los archivos CSV en los DataFrames

# Tablas de hechos
df_viajes = pd.read_csv("../data/raw/fact_viajes.csv").set_index('viaje_id')
df_incidencias = pd.read_csv("../data/raw/fact_incidencias.csv").set_index('incidencia_id')
df_mantenimientos = pd.read_csv("../data/raw/fact_mantenimiento.csv").set_index('mantenimiento_id')

# Tablas de dimensión
df_lineas = pd.read_csv("../data/raw/dim_linea.csv").set_index('linea_id')
df_vehiculos = pd.read_csv("../data/raw/dim_vehiculo.csv").set_index('vehiculo_id')
df_conductores = pd.read_csv("../data/raw/dim_conductor.csv").set_index('conductor_id')
df_paradas = pd.read_csv("../data/raw/dim_parada.csv").set_index('parada_id')
df_tarifas = pd.read_csv("../data/raw/dim_tarifa.csv").set_index('tarifa_id')
df_cocheras = pd.read_csv("../data/raw/dim_depot.csv").set_index('depot_id')

### **8.1 Viajes**

In [ ]:
df_viajes[ (df_viajes.consumo.isnull())].head(5)

In [ ]:
df_viajes[ (df_viajes.consumo.isnull())].head(5)

def calcular_media_consumo(df, km_programados, km_recorridos, vehiculo_id):
    df_consumo = df[ (df['km_programados'] == km_programados) & ( df['km_recorridos'] == km_recorridos ) & ( df['vehiculo_id'] == vehiculo_id )]['consumo']
    return np.mean(df_consumo)

# Podemos calcular el consumo del viaje haciendo la media de las filas obtenidas
for _, fila in df_viajes[ (df_viajes.consumo.isnull())].iterrows():
    df_viajes.loc[ (df_viajes.consumo.isnull()), 'consumo'] = calcular_media_consumo(df_viajes, fila['km_programados'], fila['km_recorridos'], fila['vehiculo_id'] )

In [ ]:
df_filtrado = df_viajes[df_viajes.pasajeros_subidos.isnull()]
df_inner = pd.merge(df_filtrado, df_vehiculos, on='vehiculo_id', how='left')

for _, linea in df_inner.iterrows():
    df_viajes.loc[(df_viajes.pasajeros_subidos.isnull()), 'pasajeros_subidos'] =  np.round(linea['ocupacion_pct'] * linea['capacidad_total'])

df_viajes[df_viajes.pasajeros_subidos.isnull()]

In [ ]:
# Fecha es un object no un datetime
df_viajes.fecha = pd.to_datetime(df_viajes.fecha)

In [ ]:
df_viajes.dia_semana = df_viajes.dia_semana.apply(lambda x: x.capitalize())
df_viajes.dia_semana.unique()

In [ ]:
df_viajes.ocupacion_pct = df_viajes.ocupacion_pct.apply(lambda x: x * 100)

In [ ]:
df_viajes.info()

### **8.2 Mantenimiento**

In [ ]:
# Convertir los numeros negativos en positivos
for _, fila in df_mantenimientos[ df_mantenimientos.coste_eur < 0].iterrows():
    df_mantenimientos.loc[ (df_mantenimientos.coste_eur < 0), 'coste_eur'] = abs(fila.coste_eur )

# Modificamos valore null categoria
for indice, fila in df_mantenimientos[ (df_mantenimientos.categoria.isnull())].iterrows():
    df_mantenimientos.loc[ indice,  'categoria'] = df_mantenimientos[ df_mantenimientos['tipo_mantenimiento'] == fila['tipo_mantenimiento']]['categoria'].iloc[0]

df_mantenimientos.info()

### **8.3 Incidencias**

In [ ]:
df_incidencias[ df_incidencias['coste_estimado_eur'] <= 0]['tipo_incidencia'].unique()

### **8.4 Conductores**

In [ ]:
df_conductores.loc[ df_conductores['turno_habitual'] == "manana", "turno_habitual"] = "Manana (06-14h)"
df_conductores['turno_habitual'].unique()

### **8.7 Paradas**

In [4]:
df_paradas[df_paradas['latitud'].isnull()]

,nombre_parada,barrio,tipo,latitud,longitud,accesible_silla,marquesina,panel_informacion,activa
parada_id,,,,,,,,,
11,Parada Centro 11,Centro,Intermedia,NaN,-3.711652,True,True,True,True


In [ ]:
df_paradas.dropna(subset=['latitud'], inplace=True)

Se detectó un valor nulo en la columna latitud, una variable esencial para la geolocalización de las paradas. Se descartó la imputación mediante la media de las latitudes de la categoria barrio, ya que generaría una coordenada ficticia y podría introducir errores en futuros análisis geográficos. Dado que únicamente existe un registro afectado, se optó por eliminar dicha fila para mantener la integridad del conjunto de datos.

In [ ]:
# TODO: QUE HACER CON ESTO
df_paradas[df_paradas['latitud'] > 100]

In [ ]:
df_paradas[df_paradas['accesible_silla'].isnull()]

In [ ]:
df_paradas.dropna(subset=['accesible_silla'], inplace=True)

Como no tenemos forma de saber si cuenta con acceso a silla optamos por eliminar la fila

In [ ]:
df_paradas['accesible_silla'].dtype

In [ ]:
def convertir_a_bool(fila):
    return True if fila['accesible_silla'] == True else False

df_paradas['accesible_silla'] = df_paradas.apply(convertir_a_bool, axis= 1) 

In [ ]:
df_paradas.info()

In [ ]:
df_paradas['barrio'].unique()

In [ ]:
df_paradas['barrio'] = df_paradas['barrio'].apply(lambda x: x.title()) 
df_paradas['barrio'].unique()

### **8.9 Vehículos**

In [ ]:
df_vehiculos.head()

In [ ]:
# Podemos observar como el año de fabricacion siempre es un año menor que el año de incorporacion
df_vehiculos[ df_vehiculos.anno_fabricacion - df_vehiculos.anno_incorporacion > 1]

In [ ]:
# Procedemos a modificar el año de fabricacion
proximo_anno = datetime.now().year + 1
mask =  df_vehiculos.anno_fabricacion > proximo_anno 

df_vehiculos.loc[  mask, "anno_fabricacion"] = df_vehiculos[mask]["anno_incorporacion"] - 1

In [ ]:
df_vehiculos[df_vehiculos["km_totales"] < 1000]

In [ ]:
df_vehiculos[df_vehiculos["anno_incorporacion"] == 2016]